# Listening check (LC1 to LC4)

Are the false alarms real calls nobody labelled, and are the labels every model misses real calls?
Pre-registered in `ai-interactions/preregistration.md` (LC1 to LC4). Pack made with `python -m aivis.phase1.listen pack` on the server.

**How to label**
- Every item shows the site, the call type to look for, a spectrogram with the window between two dashed lines, and the audio 10x faster (first the window, then 60 s around it).
- Below every item are **references**: labelled calls of the same type from the same site, one per sub-type (box = the label), with audio. Compare against them, especially when unsure.
- Answer **yes** if at least one call of that type is in the window, even partly; **no** if not; **unsure** only if you really cannot tell.
- If you hear a different call type (or a ship, ice, a glitch), pick it under *also heard*. Notes are optional.
- Work alone, do not talk about single items before everyone is done, and do not open `key/` or `reveal/`.
- About 110 items, roughly a minute each. Every answer is saved right away, you can stop and come back any time (it starts at the first unanswered item).
- Headphones help a lot for the 20 Hz calls.

**Setup**
1. `git pull`, and `pip install anywidget` once (for the play line on the spectrogram; without it the form still works, just without the line); the items are in the repo (`results/phase1/listening/blind`), so this runs anywhere you have the repo, laptop or server.
2. In the next cell put your first name in `ANNOTATOR`, leave `PACK = None`.
3. Run the cells and label. Your answers land in `results/phase1/listening/annotations_<your name>.csv`.
4. When you are done, send that csv to Matthias (or commit just that file). Do not run the results part below, it needs the key, which is not in git on purpose.

In [ ]:
from pathlib import Path
from aivis.phase1 import listen

ANNOTATOR = ""      # your first name, lowercase, e.g. "matthias"
PACK = None         # None = results/phase1/listening/blind from git
assert ANNOTATOR, "put your name in ANNOTATOR"


In [ ]:
listen.labeller(ANNOTATOR, PACK)

## Results

Only once everyone is done. Needs `key/` from the pack and every `annotations_<name>.csv` in `results/phase1/listening/`.

In [ ]:
out = listen.report(quiet=True)
for name in ["quality", "agreement", "shares", "precision"]:
    if name in out:
        print(name); display(out[name].round(3))

Gate for LC1 to LC3 (LC4): kappa at least 0.6, controls at least 90 % right for every annotator; with three or more annotators the kappa is Fleiss' kappa over all. Otherwise the results are reported as inconclusive.

In [ ]:
q, ag = out["quality"], out.get("agreement")
k = None
if ag is not None:
    k = ag[ag.pair == "all (Fleiss)"].kappa_3way.min() if (ag.pair == "all (Fleiss)").any() else ag.kappa_3way.min()
gate = (q.hit_said_yes.min() >= 0.9) and (q.bg_said_no.min() >= 0.9) and (k is None or k >= 0.6)
print(f"kappa {k}", "LC4 gate passed" if gate else "LC4 gate failed, LC1 to LC3 are inconclusive")

## Reveal

Items we disagree on, and false alarms we both judged real calls, with the labels and every model's events.

In [ ]:
from IPython.display import Image, display
import pandas as pd
key = pd.read_csv(listen.PACK / "key" / "items_key.csv")
A = listen.load_answers(listen.RESULTS).pivot(index="item_id", columns="annotator", values="answer")
split = A[A.nunique(axis=1) > 1].index
real_fa = key[(key.group == "fa") & key.item_id.isin(A[(A == "yes").all(axis=1)].index)].item_id
for title, ids in [("disagreements", split), ("false alarms judged real", real_fa)]:
    print(f"{title}: {len(ids)}")
    for i in list(ids)[:15]:
        print(i, dict(A.loc[i]))
        display(Image(filename=str(listen.PACK / "reveal" / f"{i}.png"), width=900))